In [12]:
import numpy as np
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from scipy.stats import multivariate_normal
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix


In [13]:
mu0 = [2, 2]
mu1 = [5, 5]
mu2 = [2, 5]

sigma = [[1, 0.5], [0.5, 1]]

n_samples_per_class = 500


np.random.seed(42) 


In [14]:
X_apples = np.dot(np.random.randn(n_samples_per_class, 2), sigma) + mu0
X_oranges = np.dot(np.random.randn(n_samples_per_class, 2), sigma) + mu1
X_bananas = np.dot(np.random.randn(n_samples_per_class, 2), sigma) + mu2

X = np.vstack((X_apples, X_oranges, X_bananas))
y = np.hstack((np.zeros(n_samples_per_class), np.ones(n_samples_per_class), 2 * np.ones(n_samples_per_class)))


train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]



scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:
mu0cap = np.mean(X_train_scaled[y_train == 0], axis=0)
mu1cap = np.mean(X_train_scaled[y_train == 1], axis=0)
mu2cap = np.mean(X_train_scaled[y_train == 2], axis=0)

sigmacap = np.cov(X_train_scaled, rowvar=False)

phi = np.bincount(y_train) / len(y_train)

TypeError: Cannot cast array data from dtype('float64') to dtype('int64') according to the rule 'safe'

In [ ]:


lda = LinearDiscriminantAnalysis()
lda.fit(X_train_scaled, y_train)

def plot_decision_regions_and_contours(X, y, lda_model, title):
    h = .02 
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    Z = lda_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(xx, yy, Z, alpha=0.8)
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k')
    plt.title(title)
    plt.xlabel('Weight')
    plt.ylabel('Surface Smootheness')
    plt.show()

for class_label in range(3):
    mu = lda.means_[class_label]
    sigma = lda.covariance_
    ellipse = Ellipse(mu, np.sqrt(sigma[0, 0]), np.sqrt(sigma[1, 1]), 
                      angle=np.degrees(np.arctan2(sigma[1, 0], sigma[0, 0])), color='r')
    
    plt.figure(figsize=(8, 6))
    plot_decision_regions_and_contours(X_train_scaled, y_train, lda, f'Decision Regions and Gaussian Contour for Class {class_label}')
    plt.gca().add_artist(ellipse)

In [ ]:
y_pred = lda.predict(X_test_scaled)

conf_matrix = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:')
print(conf_matrix)